# Baselines

A majority-class model, a logistic regression that only knows how long the narrative is, and TF-IDF with logistic regression. All three are scored with 5-fold cross-validation on the training set. The test set isn't touched until the final model is picked.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
from sklearn.metrics import confusion_matrix
from src.data import load_text, time_split
from src.models import majority, length_only, tfidf_logistic, cross_validate

In [2]:
train, test = time_split(load_text())
X_train = train["text"]
y_train = train["disputed"]
len(X_train), y_train.mean()

(112256, np.float64(0.23240628563283922))

## Folds

Template letters put the same text in the training folds and in the validation fold, and a model scores a letter it has already seen more easily than a new one. Narratives that start with the same 100 characters are kept in the same fold, the same rule as in the split notebook.

In [3]:
opening = X_train.str.lower().str[:100]
opening.nunique(), (opening.map(opening.value_counts()) > 1).sum()

(107642, np.int64(7236))

7,236 of the 112,256 training narratives share an opening with at least one other, so the grouping matters for 6.4% of them.

## Majority class

In [4]:
oof_majority, scores_majority = cross_validate(majority(), X_train, y_train, groups=opening)
scores_majority.mean()

roc_auc          0.500000
avg_precision    0.232406
dtype: float64

## Length only

In [5]:
oof_length, scores_length = cross_validate(length_only(), X_train, y_train, groups=opening)
scores_length.round(4)

,roc_auc,avg_precision
0,0.5687,0.2724
1,0.5620,0.2727
2,0.5720,0.2848
3,0.5682,0.2752
4,0.5623,0.2771


In [6]:
scores_length.mean()

roc_auc          0.566655
avg_precision    0.276463
dtype: float64

Length alone gets a ROC-AUC of 0.567 and an average precision of 0.276, against 0.232 for the majority model. The EDA showed that longer narratives are disputed more, and that's worth about 0.07 ROC-AUC before any word is read.

## TF-IDF and logistic regression

In [7]:
oof_tfidf, scores_tfidf = cross_validate(tfidf_logistic(), X_train, y_train, groups=opening)
scores_tfidf.round(4)

,roc_auc,avg_precision
0,0.6336,0.3483
1,0.6267,0.3394
2,0.6259,0.3488
3,0.6239,0.3350
4,0.6327,0.3444


In [8]:
scores_tfidf.mean()

roc_auc          0.628540
avg_precision    0.343192
dtype: float64

In [9]:
cm = confusion_matrix(y_train, oof_tfidf >= 0.5)
cm

array([[55079, 31088],
       [11904, 14185]])

In [10]:
tn, fp, fn, tp = cm.ravel()
pd.Series({"recall": tp / (tp + fn), "precision": tp / (tp + fp), "flagged": (tp + fp) / len(y_train)}).round(3)

recall       0.544
precision    0.313
flagged      0.403
dtype: float64

In [11]:
fitted = tfidf_logistic().fit(X_train, y_train)
len(fitted[0].vocabulary_)

17231

TF-IDF with logistic regression gets a ROC-AUC of 0.629 and an average precision of 0.343, with the folds between 0.624 and 0.634. Fitted on the whole training set, the vocabulary is 17,231 words. At a 0.5 cut-off it flags 40.3% of narratives and catches 54.4% of disputes, and 31.3% of the flagged ones are disputed.

## What the grouped folds change

In [12]:
_, scores_tfidf_plain = cross_validate(tfidf_logistic(), X_train, y_train)
_, scores_length_plain = cross_validate(length_only(), X_train, y_train)
pd.DataFrame({
    "tfidf, grouped": scores_tfidf.mean(),
    "tfidf, plain": scores_tfidf_plain.mean(),
    "length only, grouped": scores_length.mean(),
    "length only, plain": scores_length_plain.mean(),
}).T.round(4)

,roc_auc,avg_precision
"tfidf, grouped",0.6285,0.3432
"tfidf, plain",0.6335,0.3490
"length only, grouped",0.5667,0.2765
"length only, plain",0.5666,0.2764


With plain stratified folds the TF-IDF model scores 0.6335 and 0.3490, so letting copies into both sides adds 0.005 ROC-AUC and 0.006 average precision. The length-only model can't recognise a letter, and its score doesn't move. The grouped folds are used from here on.

## Results so far

In [13]:
results = pd.DataFrame({
    "majority": scores_majority.mean(),
    "length only": scores_length.mean(),
    "tfidf + logistic": scores_tfidf.mean(),
}).T
results.round(3)

,roc_auc,avg_precision
majority,0.500,0.232
length only,0.567,0.276
tfidf + logistic,0.629,0.343


The text model's ranked list is about 1.5 times cleaner than picking at random (0.343 against 0.232), and length explains part of that. It isn't comparable with the first project's 0.646 yet, because that was scored on all complaints and this is only the ones with a narrative. Both models get scored on the same rows later.